### This notebook displays all the tables and provides detailed explanations for each, followed by notes on how the three stages of normalization were met

# I. Main Tables

# 1. Table: movies

    Primary Key: movie_id (INTEGER, Auto-increment)
    Unique Attribute: rt_link (TEXT/VARCHAR, UNIQUE) — The original "rotten_tomatoes_link"
    used to identify records during the db_load.py process.
    Foreign Keys: company_id, rating_id
    Attributes:
        movie_title (TEXT, NOT NULL)
        movie_info (TEXT)
        critics_consensus (TEXT)
        runtime (INTEGER)
        original_release_date (DATE)
        streaming_release_date (DATE)
    Notes:
        Date Types: Crucial for Menu Item 7; using the DATE type allows SQL to calculate
        the 12-month difference between cinema and streaming releases.
        Efficiency: Using an integer movie_id makes joining tables significantly faster
        than using long text strings.

# 2. Table: movie_metrics (The Ratings Partition)

    Primary/Foreign Key: movie_id (INTEGER, links to movies.movie_id)
    Attributes:
        tomatometer_status (TEXT)
        tomatometer_rating (DECIMAL)
        tomatometer_count (INTEGER)
        audience_status (TEXT)
        audience_rating (DECIMAL)
        audience_count (INTEGER)
        top_critics_count (INTEGER)
        fresh_critics_count (INTEGER)
        rotten_critics_count (INTEGER)
    Notes:
        1-to-1 Relationship: Every movie has exactly one set of metrics.
        Menu Item 8: This table allows the database to efficiently SUM
        the three critic counts to find "review toppers" for each year.

# 3. Table: persons

    Primary Key: person_id (INTEGER, Auto-increment)
    Attribute: full_name (TEXT/VARCHAR, NOT NULL)
    Notes: This is a unified table for everyone listed as an actor, director, or writer in the CSV.
    This is required to find people who have functioned as both actor and director
     (Menu Item 10

# 4. Table: genres

    Primary Key: genre_id (INTEGER, Auto-increment)
    Attribute: genre_name (TEXT/VARCHAR, NOT NULL)
    Notes: Stores unique genre names (e.g., "Classics," "Drama")
    to eliminate the multivalued comma-separated strings found in the CSV

# 5. Table: production_companies

    Primary Key: company_id (INTEGER, Auto-increment)
    Attribute: company_name (TEXT/VARCHAR, NOT NULL)
    Notes: Necessary for Menu Item 6; eliminates repeating
    long company names like "Twentieth Century Fox" for every film row

# 6. Table: content_ratings

    Primary Key: rating_id (INTEGER, Auto-increment)
    Attribute: rating_name (TEXT/VARCHAR, NOT NULL) — e.g., "R", "PG-13", "NR".
    Notes: Essential for Menu Item 5 to find the shortest and longest films grouped by rating

# II. Junction Tables (The Many-to-Many Links)
These tables link movies to entities where one movie has many entries, and one person/genre
belongs to many movies.

## 7. Table: movie_actors

    Columns: movie_id (FK), person_id (FK)
    Notes: Connects movies to their entire cast list. Used for the count in Menu Item 2.

## 8. Table: movie_directors

    Columns: movie_id (FK), person_id (FK)
    Notes: Used to find the most active director (Menu Item 4) and cross-reference for Menu Item 10.

## 9. Table: movie_writers

    Columns: movie_id (FK), person_id (FK)
    Notes: Links the "authors" column from the CSV to the persons table. Required for the writer
    count in Menu Item 2.

## 10. Table: movie_genres

    Columns: movie_id (FK), genre_id (FK)
    Notes: Necessary for Menu Item 3 to calculate genre frequency and sort by the most frequent.

# Normalisation Notes

### First Normal Form (1NF): Eliminate Multivalued Attributes
Definition: We must ensure each cell contains only one atomic value.

    The Problem: In the provided CSV, the columns genres, directors, authors, and actors contain lists of names separated by commas (e.g., "Classics, Comedy, Drama"). This violates 1NF because a single cell holds multiple pieces of data.
    What we did:
        We planned to split these flat strings into individual records.
        We created separate entity tables: genres and persons [Discussion].
        We created Junction Tables (movie_actors, movie_genres, etc.) to link one movie to many individual actors or genres. This ensures that every entry in our database is a single, "atomic" value [Discussion].

### Second Normal Form (2NF): Eliminate Redundant Data
Definition: The database must be in 1NF, and all non-key attributes must be fully functionally dependent on the Primary Key. We must eliminate "partial dependencies" where data is repeated across many rows.

    The Problem: Attributes like production_company and content_rating are repeated as long text strings for thousands of films. Storing "Twentieth Century Fox" 500 times is redundant and risks data entry errors.
    What we did:
        We moved these repeated attributes into their own tables: production_companies and content_ratings [Discussion].
        In the main movies table, we replaced the long text strings with simple integer Foreign Keys (company_id and rating_id).
        Now, the specific name of a company is stored in exactly one place, fulfilling the 2NF requirement to eliminate redundancy [Discussion].

### Third Normal Form (3NF): Eliminate Transitive Dependencies
Definition: The database must be in 2NF, and no non-key attribute should depend on another non-key attribute (no transitive dependencies).

    The Problem: The movie metrics (like tomatometer_status, audience_rating, and critic_counts) are facts about the movie's performance, but they form a distinct group of data from the movie's "identity" (title, release date, runtime).
    What we did:
        We performed Vertical Partitioning by creating the movie_metrics table [Discussion].
        By separating the "Scores/Counts" from the "Movie Metadata," we ensured that the main movies table is not overburdened with performance stats that might change or be updated independently.
        This also makes our SQL calculations for Menu Item 8 (summing review counts) more efficient because the database only has to scan the smaller metrics table